# 05 — ResNet-18: ablação de capacidade (Kaggle)

Testa a hipótese **"menos parâmetros generaliza melhor com poucos dados"**: ResNet-18 (~11,2M params) vs ResNet-50 (~23,5M), **mantendo transfer learning** e a mesma metodologia V2 (LR diferenciado, dropout 0,3, weight_decay 0,1, augmentation strong, weighted CE, split_v1).

Notebook **enxuto e focado** — só setup + 1 treino. Não roda os treinos caros de ViT/Swin (notebook 04).

## Pré-requisitos
1. **+ Add Input** → `ninadaithal/imagesoasis`
2. **Settings → Accelerator → GPU T4** + **Internet → On**
3. **Save Version → Save & Run All (Commit)** (imune a desconexão)

## 1. Setup (GPU, dataset, repo, deps)

In [ ]:
import os, glob, sys, torch
assert torch.cuda.is_available(), 'GPU nao detectada. Settings -> Accelerator -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

hits = glob.glob('/kaggle/input/**/Non Demented', recursive=True)
assert hits, 'Adicione o dataset ninadaithal/imagesoasis em + Add Input.'
DATA_DIR = os.path.dirname(hits[0])
print('Data dir:', DATA_DIR)

%cd /kaggle/working
if not os.path.isdir('/kaggle/working/TCC'):
    !git clone https://github.com/ThiagoCB1900/TCC.git
else:
    %cd /kaggle/working/TCC
    !git fetch --all && git reset --hard origin/main
%cd /kaggle/working/TCC
!git log --oneline -1

for link, target in [('/kaggle/working/TCC/Data', DATA_DIR),
                     ('/kaggle/working/TCC/experiments/runs', '/kaggle/working/runs')]:
    os.makedirs('/kaggle/working/runs', exist_ok=True)
    if os.path.islink(link) or os.path.exists(link):
        os.system(f'rm -rf {link}')
    os.symlink(target, link)

!pip install -q timm==1.0.11 monai==1.3.2 torchmetrics==1.4.3 tabulate==0.9.0 seaborn==0.13.2
if '/kaggle/working/TCC' not in sys.path:
    sys.path.insert(0, '/kaggle/working/TCC')
print('Setup OK')

## 2. Treino ResNet-18 (weighted, V2) — ~1h

In [ ]:
!python -m src.training.run \
    --model resnet18 --epochs 20 \
    --batch-size 32 --batch-size-eval 64 --num-workers 2 \
    --run-name resnet18_class3_weighted_kaggle

## 3. Métricas + comparação com ResNet-50 (se a run estiver presente)

In [ ]:
import json
from pathlib import Path
import pandas as pd
runs_dir = Path('/kaggle/working/runs')

def latest(tag):
    c = sorted([d for d in runs_dir.iterdir() if d.is_dir() and tag in d.name and 'smoke' not in d.name])
    return c[-1] if c else None

rows = []
for label, tag in [('ResNet-18', 'resnet18_class3_weighted_kaggle'),
                   ('ResNet-50 V2', 'resnet50_v2_class3_weighted_kaggle')]:
    r = latest(tag)
    if r is None:
        print(f'(sem run {label})'); continue
    f = json.loads((r / 'final_test_metrics.json').read_text(encoding='utf-8'))
    t = f['test_metrics']
    rows.append({'modelo': label, 'best_ep': f['best_epoch'],
                 'accuracy': t['accuracy'], 'balanced_accuracy': t['balanced_accuracy'],
                 'macro_f1': t['macro_f1'], 'auc_macro': t['auc_macro'],
                 'F1_mild_or_mod': t['f1_per_class'].get('mild_or_moderate')})
print(pd.DataFrame(rows).set_index('modelo').T.to_string())

## 4. Compactar para baixar e commitar

In [ ]:
import shutil
run = latest('resnet18_class3_weighted_kaggle')
out = Path('/kaggle/working/resnet18_to_commit'); out.mkdir(exist_ok=True)
tgt = out / run.name; tgt.mkdir(exist_ok=True)
for fn in ['config.yaml','history.json','final_test_metrics.json','train.log','test_predictions.npz']:
    if (run / fn).exists(): shutil.copy(run / fn, tgt / fn)
shutil.make_archive('/kaggle/working/resnet18_to_commit', 'zip', out)
print('Pronto: /kaggle/working/resnet18_to_commit.zip')
print('Inclui test_predictions.npz -> entra no McNemar das arquiteturas.')
!ls -la /kaggle/working/resnet18_to_commit.zip